# 00 - Ingest: from the raw CSV to Parquet
**HMDA 2024 project - Group 7.** This is the first step of the project: take the instructor-provided CSV, record the dataset facts required by the assignment (format, size, number of records, number of variables), then convert it to Parquet so every later step uses the same input.

This notebook uses **PySpark** because the raw file is large (4.4 GB). Run the cells from top to bottom.

## Step 1. Mount Drive and set the paths
The raw zip `2024_public_lar_one_year_csv.zip` must be in the `hmda` folder of My Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob, time, subprocess, shutil

DRIVE_DIR     = "/content/drive/MyDrive/hmda"
ZIP_PATH      = f"{DRIVE_DIR}/2024_public_lar_one_year_csv.zip"
RAW_DIR       = "/content/hmda_raw"
PARQUET_LOCAL = "/content/hmda_parquet"
PARQUET_DRIVE = f"{DRIVE_DIR}/hmda_parquet"
RESULTS_DIR   = f"{DRIVE_DIR}/results"
FORCE_REBUILD = False        # True = rewrite the Parquet even if it already exists

assert os.path.exists(ZIP_PATH), "Zip file not found on Drive - check the hmda folder and the file name"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("OK")

Mounted at /content/drive
OK


## Step 2. Unzip and record the file facts
The assignment asks for: file format, size, number of records, number of variables.

In [2]:
if not glob.glob(f"{RAW_DIR}/**/*.csv", recursive=True):
    !unzip -q -o "{ZIP_PATH}" -d "{RAW_DIR}"

csv_path = glob.glob(f"{RAW_DIR}/**/*.csv", recursive=True)[0]
size_bytes = os.path.getsize(csv_path)
print("CSV :", csv_path)
print(f"Size: {size_bytes/1e9:.2f} GB")
print("Header (first 300 characters):")
!head -n 1 "{csv_path}" | cut -c1-300

CSV : /content/hmda_raw/2024_public_lar_one_year_csv.csv
Size: 4.64 GB
Header (first 300 characters):
activity_year,lei,derived_msa_md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,derived_race,derived_sex,action_taken,purchaser_type,preapproval,loan_type,loan_purpose,lien_status,reverse_mortgage,open_end_line_of_credi


## Step 3. Start Spark

In [3]:
!pip install -q pyspark==4.0.5   # same version as requirements.txt
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("local[*]")
         .config("spark.driver.memory", "8g")
         .appName("hmda_ingest").getOrCreate())
print("Spark", spark.version)

Spark 4.0.5


## Step 4. Read the CSV and write Parquet
All columns are read as **strings** (no `inferSchema`) on purpose: codes such as `NA`, `Exempt` and `1111` must be preserved, and type casting is done in a controlled way in the cleaning step. This step takes about 15-20 minutes.

In [4]:
t0 = time.perf_counter()
df = spark.read.csv(csv_path, header=True)          # every column is a string
n_cols = len(df.columns)

if os.path.exists(PARQUET_LOCAL) and not FORCE_REBUILD:
    print("Parquet already exists in /content, skipping the write")
else:
    df.write.mode("overwrite").parquet(PARQUET_LOCAL)
    print(f"Parquet written in {time.perf_counter()-t0:.0f}s")

Parquet written in 487s


## Step 5. Check: number of records and variables against the number of lines in the file

In [5]:
pq = spark.read.parquet(PARQUET_LOCAL)
n_rows = pq.count()
n_lines = int(subprocess.check_output(["wc", "-l", csv_path]).split()[0])

print("Records (Spark)    :", f"{n_rows:,}")
print("File lines - 1     :", f"{n_lines-1:,}   (must equal the number of records)")
print("Variables          :", len(pq.columns))
assert n_rows == n_lines - 1, "Number of records differs from the number of file lines - check again"
assert n_rows == 12259199 and len(pq.columns) == 99, "Differs from the earlier measurement (12,259,199 rows, 99 columns)"
print("=> MATCH")

Records (Spark)    : 12,259,199
File lines - 1     : 12,259,199   (must equal the number of records)
Variables          : 99
=> MATCH


## Step 6. What is one row? Distribution of application outcomes (`action_taken`)
The assignment asks us to distinguish applications, originated loans, denials and the other outcomes.

In [6]:
labels = {1: "Loan originated", 2: "Approved, not accepted", 3: "Denied", 4: "Withdrawn by applicant",
          5: "File closed (incomplete)", 6: "Purchased loan", 7: "Preapproval denied", 8: "Preapproval approved, not accepted"}
rows = pq.groupBy("action_taken").count().orderBy("action_taken").collect()
for r in rows:
    print(f"{r['action_taken']}  {labels[int(r['action_taken'])]:38s} {r['count']:>10,}  ({r['count']/n_rows:6.1%})")

1  Loan originated                         6,197,085  ( 50.6%)
2  Approved, not accepted                    361,232  (  2.9%)
3  Denied                                  2,103,462  ( 17.2%)
4  Withdrawn by applicant                  1,538,273  ( 12.5%)
5  File closed (incomplete)                  581,940  (  4.7%)
6  Purchased loan                          1,276,504  ( 10.4%)
7  Preapproval denied                         47,312  (  0.4%)
8  Preapproval approved, not accepted        153,391  (  1.3%)


## Step 7. Save the dataset facts and copy the Parquet to Drive

In [7]:
facts = {
    "source": "2024 HMDA public Loan/Application Register (CFPB/FFIEC), instructor-provided file",
    "reporting_year": 2024,
    "file_name": os.path.basename(csv_path),
    "file_format": "CSV (comma-separated, header row)",
    "size_bytes": size_bytes,
    "size_gb": round(size_bytes / 1e9, 2),
    "n_records": n_rows,
    "n_variables": len(pq.columns),
    "unit_of_observation": "one loan application or purchased loan (not necessarily a funded mortgage)",
    "all_columns_read_as_string": True,
    "columns": pq.columns,
}
with open(f"{RESULTS_DIR}/dataset_facts.json", "w") as fh:
    json.dump(facts, fh, indent=2)
print("Saved", f"{RESULTS_DIR}/dataset_facts.json")

# Copy only if the destination does NOT exist yet (otherwise cp -r would nest a folder inside it)
if os.path.exists(PARQUET_DRIVE):
    print("Parquet already on Drive, keeping it:", PARQUET_DRIVE)
else:
    shutil.copytree(PARQUET_LOCAL, PARQUET_DRIVE)
    print("Parquet copied to Drive")

Saved /content/drive/MyDrive/hmda/results/dataset_facts.json
Parquet already on Drive, keeping it: /content/drive/MyDrive/hmda/hmda_parquet
